# Classifier Performance Analysis 
How does test accuracy degrade under different threat models.
Classifier Training set covers all T3 attacks (incl.ALIE, IPM)

In [1]:
%load_ext autoreload
import sys
import copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [11]:
%autoreload 2
from lib.graph_factory import GraphFactory
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, LOGFILE, NUM_NODES
from itertools import product

In [4]:
RUN_DIR = Path().resolve()
config = BASE_CONF
config['sys']['atk_type'] = 'ALIE'
atks = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']

In [5]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], config['train']['b'], seed('graph'))
proj_const_estimator = ProjConstEstimator(config, global_dataset, rng('proj-const-estimator')) 
classifier = ByzClassifier(config, global_dataset, rng('byz-classifier'))
sys_sim = SystemSimulator(config, global_dataset, rng('ml-system-sim'), LOGFILE)

In [6]:
proj_const_estimator.configure(gf,config)
proj_const = proj_const_estimator.estimate()

In [7]:
atks = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
local_seed = seed('experiment-7')
classifier.configure(gf, config, proj_const) 

In [8]:
train_sim_data = dict()
train_sim_seed = local_seed
for atk in atks:
    print(f'Train simulating {atk}')
    X, y = classifier.simulate(atk, train_sim_seed)
    train_sim_data[atk] = { 'X': X, 'y': y}

Train simulating label_flip
Train simulating sign_flip
Train simulating gaussian
Train simulating ALIE
Train simulating IPM


In [9]:
test_sim_data = dict()
test_sim_seed = local_seed + 17
for atk in atks:
    print(f'Test simulating {atk}')
    X, y = classifier.simulate(atk, test_sim_seed)
    test_sim_data[atk] = { 'X': X, 'y': y}

Test simulating label_flip
Test simulating sign_flip
Test simulating gaussian
Test simulating ALIE
Test simulating IPM


In [22]:
# 5x5 comparison grid
rows = []
for train_atk, test_atk in product(atks, atks):
    RUN_ID = generate()
    
    X_train = train_sim_data[train_atk]['X']
    y_train = train_sim_data[train_atk]['y']
    
    X_test = test_sim_data[test_atk]['X']
    y_test = test_sim_data[test_atk]['y']
    
    classifier.init_preproc()
    X_tr = classifier.feat_pre_proc.fit_transform(X_train)
    C_specs = classifier.train_and_eval(X_tr, y_train, quantile=0.8)
    
    X_te = classifier.feat_pre_proc.transform(X_test)
    C_metrics = classifier.test(X_te, y_test)

    run_results = dict()
    run_results.update(C_specs)
    run_results.update(C_metrics)
    run_results['proj_const'] = proj_const  
    run_results['train_atk'] = train_atk
    run_results['test_atk'] = test_atk
    run_results['id'] = RUN_ID
    rows.append(run_results)

df1 = pd.DataFrame(rows)
df1.set_index('id', inplace=True)

In [24]:
# LOO Analysis
rows = []
for atk in atks:
    RUN_ID = generate()
    train_atks = list(set(atks) - set(atk))
    config['train']['train_atks'] = train_atks
    config['train']['val_atks'] = [atk]
    
    X_train, y_train, X_test, y_test = classifier.run_simulations() # handles feature transforms
    C_specs = classifier.train_and_eval(X_train, y_train, quantile=0.8)
    C_metrics = classifier.test(X_test, y_test)

    run_results = dict()
    run_results.update(C_specs)
    run_results.update(C_metrics)
    run_results['proj_const'] = proj_const  
    run_results['train_atks'] =  train_atks
    run_results['test_atks'] = atk
    run_results['id'] = RUN_ID
    rows.append(run_results)

df2 = pd.DataFrame(rows)
df2.set_index('id', inplace=True)

Traceback (most recent call last):
  File "/home/linux/env/ml/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_198235/1545610504.py", line 9, in <module>
    X_train, y_train, X_test, y_test = classifier.run_simulations() # handles feature transforms
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/linux/Documents/UoE/Dissertation/numerics/lib/classifier.py", line 132, in run_simulations
    X_C, y_C = self.simulate(atk, 1)
               ^^^^^^^^^^^^^^^^^^^^^
  File "/home/linux/Documents/UoE/Dissertation/numerics/lib/classifier.py", line 121, in simulate
    thread.join()
  File "/home/linux/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/threading.py", line 1149, in join
    self._wait_for_tstate_lock()
  File "/home/linux/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/threading.py", line 11

KeyboardInterrupt: 

In [23]:
df1

,C_fpr,C_fnr,C_tau,prevalence,tau,tn,fp,fn,tp,beta_C,gamma_C,roc_auc,avg_prec,proj_const,train_atk,test_atk
id,,,,,,,,,,,,,,,,
jkLXviF7t2p6QBXGucltd,0.100992,0.198052,0.685749,0.294118,0.685749,6078,642,543,2257,0.193929,0.095536,0.939553,0.833858,1.575567,label_flip,label_flip
OJXYzs80CrhgTJd_dzQbu,0.100992,0.198052,0.685749,0.294118,0.685749,5994,726,2175,625,0.776786,0.108036,0.585354,0.417704,1.575567,label_flip,sign_flip
7wXWFhR_UnbXDYxNfkeBQ,0.100992,0.198052,0.685749,0.294118,0.685749,5742,978,0,2800,0.000000,0.145536,1.000000,1.000000,1.575567,label_flip,gaussian
zkq_1zSb90WR_aNc2VBsE,0.100992,0.198052,0.685749,0.294118,0.685749,4458,2262,2799,1,0.999643,0.336607,0.043175,0.168730,1.575567,label_flip,ALIE
yPCqf1o9jOZsk0ETM1k-H,0.100992,0.198052,0.685749,0.294118,0.685749,5993,727,2796,4,0.998571,0.108185,0.258438,0.195635,1.575567,label_flip,IPM
obfiMnGTtzwF7Sc53sHlo,0.223625,0.198052,0.508225,0.294118,0.508225,4526,2194,930,1870,0.332143,0.326488,0.747315,0.494020,1.575567,sign_flip,label_flip
0H8eMUUYJWnw8q10EzdZS,0.223625,0.198052,0.508225,0.294118,0.508225,5178,1542,587,2213,0.209643,0.229464,0.866245,0.741148,1.575567,sign_flip,sign_flip
G1nIh4GXkfxgNm4ucnNad,0.223625,0.198052,0.508225,0.294118,0.508225,3930,2790,0,2800,0.000000,0.415179,1.000000,1.000000,1.575567,sign_flip,gaussian
mVo9rx9c4LmpeHi8iGK_K,0.223625,0.198052,0.508225,0.294118,0.508225,1898,4822,2748,52,0.981429,0.717560,0.028638,0.165184,1.575567,sign_flip,ALIE
